# Week 32, Day 4: Abstract base classes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w032d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

    def describe(self):
        return f"{type(self).__name__}: {self.area()}"

class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

class Blob(Shape):
    pass

print(Square(3).describe())
for cls in (Shape, Blob):
    try:
        cls()
        print("made a", cls.__name__)
    except TypeError:
        print("can't make a", cls.__name__)

In [ ]:
from abc import ABC, abstractmethod

class Payment(ABC):
    def __init__(self, pence):
        self.pence = pence

    @abstractmethod
    def fee(self):
        """The fee for this payment, in pence."""

    def total(self):
        return self.pence + self.fee()

class Card(Payment):
    def fee(self):
        return self.pence * 15 // 1000

class Cash(Payment):
    def fee(self):
        return 0

class Cheque(Payment):
    def fea(self):                     # a typo: fee is still missing
        return 50

print(Card(2000).total(), Cash(2000).total())
try:
    Cheque(2000)
except TypeError as err:
    print("TypeError:", err)

In [ ]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    @property
    @abstractmethod
    def wheels(self):
        ...

    def tyres_needed(self, count):
        return self.wheels * count

class Bike(Vehicle):
    wheels = 2            # a plain class attribute satisfies an abstract property

class Car(Vehicle):
    @property
    def wheels(self):
        return 4

print(Bike().tyres_needed(10), Car().tyres_needed(3))

In [ ]:
import json
from abc import ABC, abstractmethod

class Store(ABC):
    """Somewhere to keep text by key."""

    @abstractmethod
    def get(self, key):
        """The text for key; KeyError if there isn't any."""

    @abstractmethod
    def put(self, key, text):
        """Keep text under key."""

    @abstractmethod
    def keys(self):
        """All the keys, sorted."""

    def get_or(self, key, default=""):
        try:
            return self.get(key)
        except KeyError:
            return default

    def copy_to(self, other):
        for key in self.keys():
            other.put(key, self.get(key))

class MemoryStore(Store):
    def __init__(self):
        self._data = {}

    def get(self, key):
        return self._data[key]

    def put(self, key, text):
        self._data[key] = text

    def keys(self):
        return sorted(self._data)

class FileStore(Store):
    def __init__(self, path):
        self.path = path

    def _load(self):
        try:
            with open(self.path) as f:
                return json.load(f)
        except FileNotFoundError:
            return {}

    def get(self, key):
        return self._load()[key]

    def put(self, key, text):
        data = self._load()
        data[key] = text
        with open(self.path, "w") as f:
            json.dump(data, f)

    def keys(self):
        return sorted(self._load())

mem = MemoryStore()
mem.put("shopping", "yam, eggs")
mem.put("todo", "call Kemi")
disk = FileStore("notes.json")
mem.copy_to(disk)
print(disk.keys(), disk.get("todo"), disk.get_or("diary", "(none)"))

In [ ]:
from collections.abc import MutableMapping

class LowerDict(MutableMapping):
    def __init__(self, data=()):
        self._data = {}
        self.update(data)                       # free, and uses __setitem__

    def __getitem__(self, key):
        return self._data[key.lower()]

    def __setitem__(self, key, value):
        self._data[key.lower()] = value

    def __delitem__(self, key):
        del self._data[key.lower()]

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

d = LowerDict({"Lagos": 15})
d.update(LEEDS=0.8)
d.setdefault("KANO", 4)
print(dict(d), d.get("LAGOS"), "leeds" in d, d.pop("Kano"), len(d))
print(d == {"lagos": 15, "leeds": 0.8})

In [ ]:
from collections.abc import Iterable, Sized, Hashable, Callable, Sequence, Mapping

class Crate:
    def __len__(self):
        return 3

print(isinstance(Crate(), Sized), isinstance(Crate(), Iterable))
print(isinstance([1], Hashable), isinstance((1,), Hashable), isinstance(len, Callable))
print(isinstance("abc", Sequence), isinstance({}, Mapping), isinstance({}, Sequence))

In [ ]:
from collections.abc import Iterable

def as_list(names):
    if isinstance(names, str) or not isinstance(names, Iterable):
        return [names]
    return list(names)

print(as_list("Ada"), as_list(["Ada", "Bo"]), as_list(("Ada",)), as_list({"Bo"}), as_list(42))

> **Key idea.** An abstract base class inherits from `abc.ABC` and marks required methods with `@abstractmethod` (under `@property` or `@classmethod` if used); Python refuses to make any object whose class hasn't written them all, which catches missing and misspelt methods early. ABCs can also hold ordinary methods built on the abstract ones. `collections.abc` provides `Sequence`, `Mapping`, `MutableMapping`, `Set` and more: write the few required special methods and get the rest free, built on yours. `isinstance(x, Iterable)` and friends check what an object can do (remember that strings are iterable). Use ABCs for families that must keep a promise; use duck typing for simple needs.

### Exercise: An abstract shape

Make `Shape` an ABC whose `area` and `perimeter` are abstract, keeping `describe` as an ordinary method. Then write `Rect(width, height)` and `Circle(radius)` (use `math.pi`). `Shape()` itself must be impossible to make.

In [ ]:
import math
from abc import ABC, abstractmethod

class Shape:
    def area(self):
        raise NotImplementedError

    def perimeter(self):
        raise NotImplementedError

    def describe(self):
        return f"{type(self).__name__}: area {self.area():.2f}, perimeter {self.perimeter():.2f}"

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shapes():\n    \"shapes\"\n    Sh, R, C = ns[\"Shape\"], ns[\"Rect\"], ns[\"Circle\"]\n    assert R(2, 3).describe() == \"Rect: area 6.00, perimeter 10.00\" and C(1).describe() == \"Circle: area 3.14, perimeter 6.28\"\n    try:\n        Sh()\n        assert False, \"Shape() should raise TypeError\"\n    except TypeError:\n        pass\n    class Half(Sh):\n        def area(self):\n            return 1\n    try:\n        Half()\n        assert False, \"a shape without perimeter can't be made\"\n    except TypeError:\n        pass")

### Exercise: A sequence for free

Write `Book(title, chapters)` as a `Sequence` of chapter titles: write only `__getitem__` (a slice gives a new `Book` with the same title) and `__len__`, plus `__repr__` like `Book('Aké', 3 chapters)`. Everything else (`in`, `for`, `reversed`, `index`, `count`) must come from `Sequence`.

In [ ]:
from collections.abc import Sequence

class Book(Sequence):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_book():\n    \"a sequence\"\n    from collections.abc import Sequence as Seq\n    B = ns[\"Book\"]\n    b = B(\"Ak\u00e9\", [\"Ijebu\", \"Ake\", \"School\", \"Return\"])\n    assert len(b) == 4 and b[0] == \"Ijebu\" and b[-1] == \"Return\" and repr(b) == \"Book('Ak\u00e9', 4 chapters)\"\n    assert \"School\" in b and list(reversed(b)) == [\"Return\", \"School\", \"Ake\", \"Ijebu\"] and b.index(\"Ake\") == 1 and b.count(\"x\") == 0\n    part = b[1:3]\n    assert isinstance(part, B) and list(part) == [\"Ake\", \"School\"] and part.title == \"Ak\u00e9\" and isinstance(b, Seq)\n    for name in [\"__contains__\", \"__iter__\", \"index\", \"count\", \"__reversed__\"]:\n        assert name not in vars(B), f\"{name} should come from Sequence\"")

### Exercise: A checked mapping

Write `Scores`, a `MutableMapping` from player names to scores: names are stored without spaces at the ends and in title case (`" ada "` becomes `"Ada"`), and scores must be whole numbers of 0 or more (not `bool`), otherwise `ValueError`. Write only the five required methods and `__init__` (which takes an optional starting dictionary). Check that the free methods (`update`, `setdefault`, `get`, `pop`) all apply your rules.

In [ ]:
from collections.abc import MutableMapping

class Scores(MutableMapping):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scores():\n    \"scores\"\n    S = ns[\"Scores\"]\n    s = S({\" ada \": 10})\n    s[\"bo\"] = 7\n    s.update({\"CHIDI\": 3}, dami=0)\n    assert dict(s) == {\"Ada\": 10, \"Bo\": 7, \"Chidi\": 3, \"Dami\": 0} and s[\"ADA\"] == 10 and s.get(\"chidi\") == 3\n    assert s.setdefault(\"bo\", 99) == 7 and s.pop(\"dami\") == 0 and len(s) == 3 and \"ada\" in s\n    for bad in [-1, 2.5, True, \"9\"]:\n        for f in [lambda: s.__setitem__(\"x\", bad), lambda: s.update(x=bad), lambda: s.setdefault(\"zed\", bad), lambda: S({\"q\": bad})]:\n            try:\n                f()\n                assert False, f\"score {bad!r} should raise ValueError\"\n            except ValueError:\n                pass\n    assert \"Zed\" not in s and \"X\" not in s\n    for name in [\"update\", \"setdefault\", \"get\", \"pop\", \"__contains__\"]:\n        assert name not in vars(S), f\"{name} should come from MutableMapping\"")

### Exercise: Flattening anything

Write `flatten(items)`, which returns a flat list of everything inside `items`, going into any nested iterable (lists, tuples, sets, ranges, your own iterable classes) but treating strings and bytes as single items. Use `collections.abc.Iterable`, not a list of types.

In [ ]:
from collections.abc import Iterable

def flatten(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_flatten():\n    \"flattening\"\n    assert flatten([1, [2, [3, (4, 5)]], 6]) == [1, 2, 3, 4, 5, 6]\n    assert flatten([\"ab\", [\"cd\", (\"ef\",)]]) == [\"ab\", \"cd\", \"ef\"] and flatten([b\"xy\", [b\"z\"]]) == [b\"xy\", b\"z\"]\n    assert flatten([range(3), [], [[[]]]]) == [0, 1, 2] and flatten([]) == []\n    class Pair:\n        def __iter__(self):\n            return iter([7, [8]])\n    assert flatten([Pair(), 9]) == [7, 8, 9]")

### Bug hunt: Can't make a circle

Making a `Circle` crashes with `TypeError: Can't instantiate abstract class Circle without an implementation for abstract method 'area'`. Fix `Circle` so this prints `12.57`.

In [ ]:
import math
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def aera(self):
        return math.pi * self.radius ** 2

print(round(Circle(2).area(), 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the area\"\n    expect_output(\"12.57\")")

### Bug hunt: Letters everywhere

`tags` should accept one tag or a list of tags, so this should print `['news', 'sport', 'news']`. But the single tag is split into letters. Fix `tags`.

In [ ]:
from collections.abc import Iterable

def tags(*groups):
    result = []
    for g in groups:
        if isinstance(g, Iterable):
            result.extend(g)
        else:
            result.append(g)
    return result

print(tags("news", ["sport", "news"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tags\"\n    expect_output(\"['news', 'sport', 'news']\")")

### Bug hunt: Half a sequence

`Week` inherits from `Sequence`, but making one crashes with `TypeError: Can't instantiate abstract class Week without an implementation for abstract method '__len__'`. Fix it so this prints `7 True 4`.

In [ ]:
from collections.abc import Sequence

class Week(Sequence):
    DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    def __getitem__(self, i):
        return Week.DAYS[i]

w = Week()
print(len(w), "Sat" in w, w.index("Fri"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a full sequence\"\n    expect_output(\"7 True 4\")")

### Puzzle: An abstract notifier

Arrange the lines into an abstract `Notifier` that requires `send`, with an ordinary `send_all` built on it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from abc import ABC, abstractmethod
from collections.abc import Iterable, Sized, Sequence

class A(ABC):
    @abstractmethod
    def f(self):
        ...

class B(A):
    def f(self):
        return 1

class C:
    def __iter__(self):
        return iter([])

p = ___   # B().f()
q = ___   # isinstance(B(), A)
r = ___   # isinstance(C(), Iterable)
s = ___   # isinstance(C(), Sized)
t = ___   # isinstance("hi", Sequence)
u = ___   # isinstance(5, Iterable)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (1, True, True), \"p to r\"\n    assert (s, t, u) == (False, True, False)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: files and folders

Model a small file system with an ABC and a mapping.

- `Node(ABC)`, made with a `name` that isn't empty and contains no `/` (otherwise `ValueError`). It has an attribute `parent` (starting as `None`), an abstract method `size()`, and an ordinary method `path()`: `"/" + name` for a node with no parent, otherwise the parent's path, `"/"`, and the name (`/home/docs/cv.txt`).
- `File(name, size_bytes)`, a `Node` whose size is `size_bytes` (an `int` of 0 or more, otherwise `ValueError`).
- `Folder(name)`, which is a `Node` **and** a `Mapping` from child names to child nodes (inherit from both). `add(node)` puts a node in the folder, sets its `parent`, and returns it; adding something that isn't a `Node` raises `TypeError`, and a name already in the folder raises `ValueError`. Iterating gives child names in alphabetical order. `size()` is the total size of everything inside, however deep.
- `Folder.find(suffix)`: the paths of every file anywhere inside whose name ends with `suffix`, sorted.

Remember that an empty folder has a length of 0, so it's false in an `if`: test `is None` where you mean "no parent".

In [ ]:
from abc import ABC, abstractmethod
from collections.abc import Mapping

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build():\n    F, D = ns[\"File\"], ns[\"Folder\"]\n    home = D(\"home\")\n    docs = home.add(D(\"docs\"))\n    docs.add(F(\"cv.txt\", 1200))\n    docs.add(F(\"notes.md\", 300))\n    pics = home.add(D(\"pics\"))\n    pics.add(F(\"lagos.jpg\", 50000))\n    trip = pics.add(D(\"trip\"))\n    trip.add(F(\"kano.jpg\", 70000))\n    trip.add(F(\"plan.txt\", 100))\n    home.add(D(\"empty\"))\n    return home\ndef test_tree():\n    \"the tree\"\n    home = build()\n    assert list(home) == [\"docs\", \"empty\", \"pics\"] and len(home) == 3 and home[\"pics\"][\"trip\"][\"kano.jpg\"].size() == 70000\n    assert home.size() == 121600 and home[\"docs\"].size() == 1500 and home[\"empty\"].size() == 0\n    assert home[\"pics\"][\"trip\"][\"plan.txt\"].path() == \"/home/pics/trip/plan.txt\" and home.path() == \"/home\"\n    assert home[\"pics\"][\"trip\"].parent is home[\"pics\"] and \"docs\" in home and home.get(\"music\") is None\n    assert home.find(\".txt\") == [\"/home/docs/cv.txt\", \"/home/pics/trip/plan.txt\"] and home.find(\".jpg\") == [\"/home/pics/lagos.jpg\", \"/home/pics/trip/kano.jpg\"]\n    assert home[\"empty\"].find(\"\") == [] and home.find(\".doc\") == []\ndef test_rules():\n    \"rules\"\n    N, F, D = ns[\"Node\"], ns[\"File\"], ns[\"Folder\"]\n    from collections.abc import Mapping as Map\n    assert issubclass(D, N) and issubclass(D, Map) and issubclass(F, N)\n    for bad in [lambda: N(\"x\"), lambda: F(\"\", 1), lambda: F(\"a/b\", 1), lambda: F(\"x\", -1), lambda: D(\"a/b\")]:\n        try:\n            bad()\n            assert False, \"should raise\"\n        except (TypeError, ValueError):\n            pass\n    d = D(\"d\")\n    d.add(F(\"a\", 1))\n    try:\n        d.add(F(\"a\", 2))\n        assert False, \"a duplicate name should raise ValueError\"\n    except ValueError:\n        pass\n    try:\n        d.add(\"a file\")\n        assert False, \"adding a non-Node should raise TypeError\"\n    except TypeError:\n        pass\n    sub = D(\"sub\")\n    d.add(sub)\n    sub.add(F(\"z\", 5))\n    assert sub[\"z\"].path() == \"/d/sub/z\" and d.size() == 6")

## Warm-up

`Shape` says every shape must have an `area`. Python then refuses to make a `Shape` itself, or a `Blob`, which never wrote one. A `Square` that keeps the rule works, and gets `describe` for free. The check happens when the object is **made**, not later when someone happens to call `area`.

## Abstract methods

An **abstract base class** is a class that exists to be inherited from and that lists methods its children **must** provide. Inherit from `abc.ABC` and mark the required methods with `@abstractmethod`:

- An abstract method's body is usually just a docstring, or `...` (the "ellipsis", Python's way of writing "nothing here yet"). It can also hold real code that children reach with `super()`.
- A class with **any** abstract method that hasn't been written can't be made. That catches a misspelt method name the moment the class is used, as `Cheque` shows. Compare Week 30's `raise NotImplementedError`, which only failed when the method was called, perhaps much later.
- ABCs can have ordinary methods (`total`), `__init__`, class attributes and everything else. A good ABC is usually a parent class from Week 30, Day 3: it lays out the job, and the abstract methods are the hooks.

Properties and class methods can be abstract too; put `@abstractmethod` **underneath** the other decorator:

## Designing a small interface

The abstract methods of an ABC are an **interface**: a promise about what every child can do. Code that works with the parent can then rely on it. Keep interfaces small: every abstract method is work for every child, now and later.

Here's a design with two different storage places for notes, sharing one interface and one set of helpers:

`copy_to` works between **any** two stores, including kinds written next year, because it only uses the interface. That's polymorphism (Week 30, Day 4) with a contract written down.

## collections.abc

The `collections.abc` module holds ABCs for Python's own kinds of container. Each one lists a few abstract methods, and gives you many more methods for free once you've written them. Week 31 promised exactly this:

| Inherit from | You write | You get for free |
|---|---|---|
| `Sequence` | `__getitem__`, `__len__` | `__contains__`, `__iter__`, `__reversed__`, `index`, `count` |
| `MutableSequence` | those, plus `__setitem__`, `__delitem__`, `insert` | `append`, `extend`, `pop`, `remove`, `reverse`, `+=`, ... |
| `Mapping` | `__getitem__`, `__iter__`, `__len__` | `__contains__`, `get`, `keys`, `items`, `values`, `__eq__` |
| `MutableMapping` | those, plus `__setitem__`, `__delitem__` | `pop`, `popitem`, `clear`, `update`, `setdefault` |
| `Set` | `__contains__`, `__iter__`, `__len__` | `&`, `|`, `-`, `^`, `<=`, `isdisjoint`, `==`, ... |

Here's the case-insensitive dictionary from Weeks 30 and 31 one last time. As a `MutableMapping`, it needs five methods, and every other dictionary method comes from the ABC, built on top of yours. So `update` and `setdefault` go through your `__setitem__`, unlike the `dict` subclass of Week 30:

The free methods are written in terms of yours, so they're sometimes slower than hand-written ones (`__contains__` from `Sequence` checks items one by one). Override any of them when it matters.

## Checking kinds with isinstance

ABCs also name **kinds** of object, which makes them the right thing to check with `isinstance`. For the simplest ones (`Iterable`, `Sized`, `Container`, `Hashable`, `Callable`), Python checks whether the object's class has the needed method, so classes count without inheriting anything:

The bigger ones (`Sequence`, `Mapping`) only count classes that inherit from them, or that are **registered** with `Sequence.register(SomeClass)`; Python's own `list`, `tuple`, `str`, `range` and `dict` are.

A classic use is a function that accepts "one thing or many". Watch out for strings: a string is iterable, so it must be checked first.

## When to use an ABC

ABCs earn their place when there's a **family** of classes that must all keep the same promise, and mistakes would be costly or late: payment types, storage backends, exporters, plugins that other people write. They also make the interface visible in one place, with docstrings, for anyone writing a new child.

They're unnecessary when a function just needs "something with a `write` method": plain duck typing (Week 30, Day 4) already handles that, and forcing callers to inherit from your class would be a nuisance. Tomorrow's Protocols describe such interfaces **without** inheritance. And `collections.abc` is worth reaching for whenever you write your own container.

## Practice

## Quiz

## Daily challenge

## Before you go

Look back at a family of classes from Week 30 (shapes, payments, exporters) and decide which methods should be abstract. Tomorrow: Protocols, which describe what an object must be able to do without asking it to inherit anything.